<h1 style="text-align:center">Exploratory Data Analysis (EDA)</h1>

## 1. Environment Setup

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

## 2. Project Paths

PROJECT_ROOT = Path().resolve().parent
DATASET_PATH = PROJECT_ROOT / "data" / "raw"
MANIFEST_PATH = PROJECT_ROOT / "data" / "dataset_manifest.csv"

## 3. Load Dataset Manifest

df = pd.read_csv(MANIFEST_PATH)

df.head()

## 4. Dataset Overview

print("Total Samples:", len(df))
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())

## 5. Class Mapping (Tumor Types)

classes = sorted(df["label"].unique())
classes

## 6. Class Distribution (Core Medical Imbalance Check)

plt.figure(figsize=(10,5))

sns.countplot(
    data=df,
    x="label",
    hue="label",
    palette="viridis",
    legend=False
)

plt.title("Tumor Type Distribution")
plt.xlabel("Tumor Type")
plt.ylabel("Number of MRI Scans")
plt.xticks(rotation=0)

plt.show()

## 7. Percentage Distribution

(df["label"].value_counts(normalize=True) * 100).round(2)

## 8. Random Sample Visualization (Per Class)

plt.figure(figsize=(12,10))

for i, cls in enumerate(classes):
    sample_path = df[df["label"] == cls]["image_path"].sample(1).values[0]
    img = Image.open(sample_path)

    plt.subplot(2,2,i+1)
    plt.imshow(img, cmap="gray")
    plt.title(f"Tumor Type: {cls}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 9. Grid Visualization (Multi-sample per Class)

fig, axes = plt.subplots(len(classes), 3, figsize=(12,12))

for i, cls in enumerate(classes):
    samples = df[df["label"] == cls]["image_path"].sample(3).values

    for j, path in enumerate(samples):
        img = Image.open(path)

        axes[i, j].imshow(img, cmap="gray")
        axes[i, j].axis("off")

        if j == 1:
            axes[i, j].set_title(cls)

plt.tight_layout()
plt.show()

## 10. Image Dimension Analysis

shapes = []

sample_df = df.sample(200, random_state=42)

for path in sample_df["image_path"]:
    img = Image.open(path)
    shapes.append(np.array(img).shape)

pd.Series(shapes).value_counts()

## 11. Pixel Intensity Distribution (Per Tumor Type)

fig, axes = plt.subplots(2,2, figsize=(12,10))
axes = axes.ravel()

for i, cls in enumerate(classes):
    sample_path = df[df["label"] == cls]["image_path"].sample(1).values[0]
    img = Image.open(sample_path).convert("L")
    arr = np.array(img)

    axes[i].hist(arr.flatten(), bins=50)
    axes[i].set_title(f"Pixel Intensity Distribution — {cls}")
    axes[i].set_xlabel("Intensity")
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 12. Mean Intensity per Class

mean_intensity = df.groupby("label")["image_path"].apply(
    lambda x: np.mean([np.array(Image.open(i).convert("L")).mean() for i in x.sample(50, random_state=42)])
)

mean_intensity

## 13. Mean MRI Representation per Class

def compute_mean_image(class_name):
    paths = df[df["label"] == class_name]["image_path"].sample(50, random_state=42)

    acc = None

    for p in paths:
        img = np.array(Image.open(p).convert("L"))
        img = np.resize(img, (224,224))

        if acc is None:
            acc = img.astype(np.float32)
        else:
            acc += img

    return acc / len(paths)


plt.figure(figsize=(12,6))

for i, cls in enumerate(classes):
    plt.subplot(1,4,i+1)
    plt.imshow(compute_mean_image(cls), cmap="gray")
    plt.title(f"Mean MRI — {cls}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 14. Corrupted Image Check

corrupted = []

for path in df["image_path"]:
    try:
        img = Image.open(path)
        img.verify()
    except:
        corrupted.append(path)

print("Corrupted Images:", len(corrupted))

## 15. Clean Dataset Creation

df_clean = df[~df["image_path"].isin(corrupted)].reset_index(drop=True)
df = df_clean
print("Original:", len(df))
print("Cleaned:", len(df_clean))

## 16. Brightness Distribution Analysis

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df["brightness"] = df["image_path"].apply(
    lambda x: np.array(Image.open(x).convert("L")).mean()
)

plt.figure(figsize=(12,6))

sns.boxplot(
    data=df_clean,
    x="label",
    y="brightness"
)

plt.title("Brightness Distribution by Tumor Type", fontsize=16)
plt.xlabel("Tumor Type", fontsize=13)
plt.ylabel("Average Pixel Brightness", fontsize=13)
plt.show()

## 17. Outlier Detection (Brightness-Based)

Q1 = df["brightness"].quantile(0.25)
Q3 = df["brightness"].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df["brightness"] < Q1 - 1.5*IQR) | (df["brightness"] > Q3 + 1.5*IQR)]

print("Outliers detected:", len(outliers))

## 18. Final Class Distribution (Cleaned)

plt.figure(figsize=(10,5))

sns.countplot(
    data=df_clean,
    x="label",
    hue="label",
    order=df_clean["label"].value_counts().index,
    palette="magma",
    legend=False
)

plt.title("Clean Dataset Class Distribution")
plt.xlabel("Tumor Type")
plt.ylabel("Count")

plt.show()

## 19. Dataset Quality Summary

summary = {
    "Total Images": len(df_clean),
    "Classes": len(df_clean["label"].unique()),
    "Corrupted Removed": len(df) - len(df_clean),
    "Most Common Class": df_clean["label"].value_counts().idxmax(),
    "Least Common Class": df_clean["label"].value_counts().idxmin()
}

summary

